In [2]:
import numpy as np
import matplotlib.pyplot as plt
import numpy.linalg as la

from boed.priors.kernels import Gaussian, Matern32
from boed.priors.gp_priors import GaussianProcessPrior
from boed.core.noise import NoiseModel,ColoredNoise
from boed.core import make_u0
from boed.utils.observation import build_selection_matrices


## Design of Single experiment
$$ y(\theta,d) = G(\theta,d) + \epsilon = \theta^{3} d^{2} + exp(-|0.2-d|) + \epsilon$$

d : design, $\theta$ : parameter, $\epsilon$ : noise 

uniform prior on [0,1]

gaussian noise 

In [6]:
def single_experiment(theta : float, d : float) -> float : 
    return theta**3 * d**2 + theta*np.exp(-np.abs(0.2 - d))

In [30]:
def jacobian_single_experiment(theta : float, d : float) -> float : 
    return 3*theta**2 * d**2 + np.exp(-np.abs(0.2 - d))

In [34]:
def jacobian_fd(f, theta : float, d : float,epsilon : float) -> float : 
    return (1./(2*epsilon))*(f(theta=theta+epsilon, d=d) - f(theta=theta-epsilon, d=d))

In [36]:
jac = jacobian_single_experiment(theta=0.2, d=0.5)
jac_approx = jacobian_fd(single_experiment, theta=0.2, d=0.5, epsilon=1e-4)
print(f"jac = {jac}")
print(f"jac approx = {jac_approx}")
print(f"error = {abs(jac_approx-jac)*100} %")

jac = 0.7708182206817179
jac approx = 0.7708182231817851
error = 2.5000671532993124e-07 %


## Calcul de Sigma y 

$\Sigma_y = \Sigma_{obs} + Cov(u(\theta,d))$, en 1D $\Sigma_y = \sigma^2 + Cov(u(\theta,d))$

In [25]:
def estimate_cov(u, d: float, n_samples: int, sigma2 = 1.0):
    theta = np.random.uniform(0, 1, n_samples)
    u_vals = np.array([u(t, d) for t in theta])
    return (1/sigma2)*(np.mean(u_vals**2) - np.mean(u_vals)**2)

### Etudier la convergence de la covariance de u($\theta$)

In [38]:
def mc_error(u, d, n_samples, sigma2 = 1.0, n_repeat=30):
    vals = np.array([estimate_cov(u, d, n_samples, sigma2) for _ in range(n_repeat)])
    mean_est = np.mean(vals)
    std_est = np.std(vals)
    rel_error = std_est / abs(mean_est)
    return mean_est, std_est, rel_error

In [33]:
n_samples_list = [100, 500, 1000, 2000, 5000, 10000, 20000]

for n in n_samples_list:
    mean_est, std_est, rel_error = mc_error(single_experiment, 0.2, n)
    print(f"n={n}, mean={mean_est:.5f}, std={std_est:.5f}, rel_error={rel_error:.3%}")

n=100, mean=0.08720, std=0.00788, rel_error=9.042%
n=500, mean=0.08969, std=0.00290, rel_error=3.233%
n=1000, mean=0.08931, std=0.00270, rel_error=3.025%
n=2000, mean=0.08970, std=0.00170, rel_error=1.899%
n=5000, mean=0.08959, std=0.00139, rel_error=1.547%
n=10000, mean=0.08955, std=0.00083, rel_error=0.930%
n=20000, mean=0.08958, std=0.00053, rel_error=0.595%


On a une bonne approximation à n = 10000, au délà le gain est marginal

### Calcul de $\Sigma_y$

In [34]:
def sigma_y(sigma2, cov, u, d, n_samples=10000) : 
    return sigma2 + cov(u, d, n_samples, n_repeat=30)[0]

In [35]:
print(sigma_y(0.001, mc_error, single_experiment, d=0.2))

0.09044979852270699


### Calcul de $l_{\theta} (u) = \mathbb{E}(Jac_{\theta} u(\theta))$, en 1d $l_{\theta} (u) = \mathbb{E}(u'(\theta))$

In [36]:
def estimate_expected_jacobian(u, d: float, n_samples: int):
    theta = np.random.uniform(0, 1, n_samples)
    u_vals = np.array([u(t, d) for t in theta])
    return np.mean(u_vals)

In [37]:
n_samples_list = [100, 500, 1000, 2000, 5000, 10000, 20000]

for n in n_samples_list:
    mean_est, std_est, rel_error = mc_error(jacobian_single_experiment, 0.2, n)
    print(f"n={n}, mean={mean_est:.5f}, std={std_est:.5f}, rel_error={rel_error:.3%}")

n=100, mean=0.00056, std=0.00007, rel_error=11.744%
n=500, mean=0.00056, std=0.00002, rel_error=4.260%
n=1000, mean=0.00057, std=0.00002, rel_error=3.274%
n=2000, mean=0.00056, std=0.00001, rel_error=2.053%
n=5000, mean=0.00057, std=0.00001, rel_error=1.366%
n=10000, mean=0.00057, std=0.00001, rel_error=1.113%
n=20000, mean=0.00057, std=0.00000, rel_error=0.666%


### Calcul de $H_{\theta}^{\Sigma_{obs}} = Cov(Jac_{\theta} u(\theta) \Sigma_{obs}^{-1/2})$ en 1D $H_{\theta}^{\Sigma_{obs}} = Cov(u'(\theta) \frac{1}{\sigma})$

In [39]:
def H_sigma_obs_theta(cov, u, d, n_samples, sigma2) : 
    return cov(u, d,n_samples,sigma2)

In [41]:
n_samples_list = [100, 500, 1000, 2000, 5000, 10000, 20000]

for n in n_samples_list:
    mean_est= H_sigma_obs_theta(estimate_cov,jacobian_single_experiment, 0.2, n,sigma2=0.001)
    print(f"n={n}, mean={mean_est:.5f}, std={std_est:.5f}, rel_error={rel_error:.3%}")

n=100, mean=0.52951, std=0.00000, rel_error=0.666%
n=500, mean=0.61405, std=0.00000, rel_error=0.666%
n=1000, mean=0.59158, std=0.00000, rel_error=0.666%
n=2000, mean=0.58463, std=0.00000, rel_error=0.666%
n=5000, mean=0.55876, std=0.00000, rel_error=0.666%
n=10000, mean=0.56852, std=0.00000, rel_error=0.666%
n=20000, mean=0.56881, std=0.00000, rel_error=0.666%


In [1]:
import numpy as np


# ── 1. Forward model (boîte noire — à remplacer) ─────────────────────────────

def u(theta):
    """
    Forward model u : R^d -> R^m.
    Remplace cette fonction par ton vrai solveur.
    """
    # Exemple jouet non linéaire
    return np.array([
        np.sin(theta[0]) + theta[1] ** 2,
        np.exp(-theta[0]) * theta[1],
    ])


# ── 2. Log-postérieure (à adapter) ───────────────────────────────────────────

def log_posterior(theta, y_obs, sigma_noise=1.0, sigma_prior=10.0):
    """
    log π(θ | y) ∝ log-vraisemblance + log-prior.
    Prior gaussien centré, likelihood gaussienne.
    """
    residual = y_obs - u(theta)
    log_lik  = -0.5 * np.dot(residual, residual) / sigma_noise**2
    log_pri  = -0.5 * np.dot(theta, theta) / sigma_prior**2
    return log_lik + log_pri


# ── 3. Jacobien par différences finies centrées ───────────────────────────────

def jacobian_fd(u, theta, h=None):
    """
    Estime J(θ) ∈ R^{m x d} par différences finies centrées.

    Coût : 2d appels à u.

    Paramètres
    ----------
    u     : callable, R^d -> R^m
    theta : array (d,)
    h     : pas de différentiation (auto si None)

    Retourne
    --------
    J : array (m, d)
    """
    d = theta.shape[0]

    if h is None:
        # Heuristique classique pour différences centrées
        h = np.finfo(float).eps ** (1/3) * (np.linalg.norm(theta) + 1e-8)

    u0 = u(theta)
    m  = u0.shape[0]
    J  = np.zeros((m, d))

    for j in range(d):
        e_j         = np.zeros(d)
        e_j[j]      = 1.0
        J[:, j]     = (u(theta + h * e_j) - u(theta - h * e_j)) / (2 * h)

    return J


# ── 4. Sampler Metropolis-Hastings (marche aléatoire) ────────────────────────

def metropolis_hastings(log_post, theta_init, n_samples, proposal_std=0.1, burnin=500):
    """
    Random-walk Metropolis-Hastings.

    Retourne
    --------
    samples : array (n_samples, d)
    acc_rate : float
    """
    d          = theta_init.shape[0]
    samples    = np.zeros((n_samples, d))
    theta_curr = theta_init.copy()
    lp_curr    = log_post(theta_curr)
    n_accept   = 0

    for i in range(-burnin, n_samples):
        theta_prop = theta_curr + proposal_std * np.random.randn(d)
        lp_prop    = log_post(theta_prop)

        log_alpha  = lp_prop - lp_curr
        if np.log(np.random.uniform()) < log_alpha:
            theta_curr = theta_prop
            lp_curr    = lp_prop
            if i >= 0:
                n_accept += 1

        if i >= 0:
            samples[i] = theta_curr

    acc_rate = n_accept / n_samples
    return samples, acc_rate


# ── 5. Estimation de E[J] et Cov[J] par MCMC ─────────────────────────────────

def estimate_jacobian_stats(u, samples, h=None):
    """
    Estime la moyenne et la covariance du Jacobien sur les échantillons MCMC.

    Paramètres
    ----------
    u       : callable, R^d -> R^m
    samples : array (K, d)
    h       : pas de différentiation (auto si None)

    Retourne
    --------
    J_mean : array (m, d)          — E_π[J(θ)]
    J_cov  : array (m*d, m*d)      — Cov_π[vec(J(θ))]
    J_all  : array (K, m, d)       — tous les Jacobiens (pour diagnostic)
    """
    K, d   = samples.shape
    m      = u(samples[0]).shape[0]
    J_all  = np.zeros((K, m, d))

    for k, theta_k in enumerate(samples):
        J_all[k] = jacobian_fd(u, theta_k, h=h)
        if (k + 1) % max(1, K // 10) == 0:
            print(f"  Jacobien {k+1}/{K} calculé")

    # Moyenne : (m, d)
    J_mean = J_all.mean(axis=0)

    # Covariance sur vec(J) : (m*d, m*d)
    J_flat = J_all.reshape(K, m * d)         # (K, m*d)
    J_cov  = np.cov(J_flat, rowvar=False)    # (m*d, m*d)

    return J_mean, J_cov, J_all


# ── 6. Pipeline principal ─────────────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)

    # Dimensions
    d = 2   # dim paramètre
    m = 2   # dim observation

    # Données synthétiques
    theta_true = np.array([0.5, -1.0])
    y_obs      = u(theta_true) + 0.1 * np.random.randn(m)

    # Log-postérieure partielle
    log_post = lambda theta: log_posterior(theta, y_obs)

    # ── MCMC
    print("=== Metropolis-Hastings ===")
    theta_init  = np.zeros(d)
    samples, ar = metropolis_hastings(log_post, theta_init,
                                      n_samples=500, proposal_std=0.3, burnin=200)
    print(f"Taux d'acceptation : {ar:.2%}")

    # ── Estimation des stats du Jacobien
    print("\n=== Estimation E[J] et Cov[J] ===")
    J_mean, J_cov, J_all = estimate_jacobian_stats(u, samples)

    print(f"\nE[J(θ)]  (shape {J_mean.shape}) :")
    print(J_mean)

    print(f"\nCov[vec(J(θ))]  (shape {J_cov.shape}) :")
    print(J_cov)

    # ── Comparaison avec le Jacobien au MAP (approximation de Laplace)
    theta_map   = samples[np.argmax([log_post(s) for s in samples])]
    J_map       = jacobian_fd(u, theta_map)
    print(f"\nJ au MAP :")
    print(J_map)

    print(f"\nDifférence ||E[J] - J_MAP|| = {np.linalg.norm(J_mean - J_map):.4f}")

=== Metropolis-Hastings ===
Taux d'acceptation : 87.20%

=== Estimation E[J] et Cov[J] ===
  Jacobien 50/500 calculé
  Jacobien 100/500 calculé
  Jacobien 150/500 calculé
  Jacobien 200/500 calculé
  Jacobien 250/500 calculé
  Jacobien 300/500 calculé
  Jacobien 350/500 calculé
  Jacobien 400/500 calculé
  Jacobien 450/500 calculé
  Jacobien 500/500 calculé

E[J(θ)]  (shape (2, 2)) :
[[-0.0956927   0.40345129]
 [ 0.02158768  0.26625708]]

Cov[vec(J(θ))]  (shape (4, 4)) :
[[ 0.35289463 -0.29682345  0.07013027  0.12911726]
 [-0.29682345  2.7591304  -0.34931309 -0.15089893]
 [ 0.07013027 -0.34931309  0.07593741  0.0483981 ]
 [ 0.12911726 -0.15089893  0.0483981   0.07385203]]

J au MAP :
[[ 0.9081296  -2.0041431 ]
 [ 0.65055332  0.64920845]]

Différence ||E[J] - J_MAP|| = 2.7104
